# L76 · 🎨 相似地图 — 音乐特征可视化

**目标**：可视化音乐特征空间——chroma 热力图读出和弦进行、t-SNE 把高维 embedding 压到2D看聚类效果、对比学习前后分布的差异。

🔗 Aurora 连接：本节是 `aurora.music` 交付物的视觉展示层，对应 `src/aurora/music/chroma.py`（chroma 提取）和 `src/aurora/music/embed.py`（对比学习 embedding）。

音乐理解的核心问题是：**两段音频有多「像」**。
Chroma 把频谱折叠到12个音高格，让和弦结构一眼可见；
Embedding 把整首曲子压成一个向量，让「像」变成欧氏距离；
t-SNE 把这些向量可视化到2D平面，验证模型是否真的学到了音乐相似性。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

## 1. Chroma 钢琴卷帘

Chroma（色度特征）把所有频率按音高的**八度等价性**折叠成12个格（C, C#, D, … B）。
公式：频率 `f` 对应的音高类别 `p = round(12 * log2(f / 440)) mod 12`。

可视化规则：横轴 = 时间帧，纵轴 = 12音高（0=C … 11=B），颜色亮度 = 该帧该音高的能量。
读图技巧：某时刻同时亮起 C(0) + E(4) + G(7) → 大三和弦；C(0) + Eb(3) + G(7) → 小三和弦。

In [ ]:
# 模拟一段「C大调 I-IV-V-I」和弦进行的 chroma 矩阵
# 每个和弦持续 16 帧：C大(0,4,7) → F大(5,9,0) → G大(7,11,2) → C大(0,4,7)
T = 64  # 总帧数
chroma = np.zeros((12, T))

chord_sequence = [
    ([0, 4, 7], 0, 16),    # C 大三和弦
    ([5, 9, 0], 16, 32),   # F 大三和弦
    ([7, 11, 2], 32, 48),  # G 大三和弦
    ([0, 4, 7], 48, 64),   # C 大三和弦（回归）
]

for notes, t_start, t_end in chord_sequence:
    for n in notes:
        chroma[n, t_start:t_end] = 1.0 + 0.15 * np.random.randn(t_end - t_start)

chroma = np.clip(chroma + 0.05 * np.abs(np.random.randn(*chroma.shape)), 0, 1.5)

pitch_names = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(chroma, aspect='auto', origin='lower', cmap='magma',
               vmin=0, vmax=1.5, interpolation='nearest')
ax.set_yticks(range(12))
ax.set_yticklabels(pitch_names, fontsize=9)
ax.set_xlabel('时间帧')
ax.set_ylabel('音高类别')
ax.set_title('Chroma 钢琴卷帘 — C大调 I-IV-V-I 和弦进行')
for label, t_mid in [('C', 8), ('F', 24), ('G', 40), ('C', 56)]:
    ax.text(t_mid, 12.2, label, ha='center', va='bottom',
            color='white', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax, label='能量')
plt.tight_layout()
plt.show()
print('✅ Chroma 图：竖向亮条 = 和弦音符同时激活，横向读和弦进行')

## 2. Embedding 与 t-SNE 降维

对比学习（Contrastive Learning）把每首曲子编码成 `d` 维向量，目标是让**同类**音乐的向量距离近、**不同类**的距离远。
损失函数（InfoNCE）：`L = -log( exp(sim(q,k+)/τ) / Σ exp(sim(q,ki)/τ) )`，其中 `sim` 为余弦相似度，`τ` 为温度超参。

t-SNE 把高维向量降到2D，保留局部邻域结构：
在高维空间计算成对相似度概率分布 `p_ij`，在低维空间用 Student-t 分布 `q_ij` 匹配，最小化 KL 散度 `KL(P || Q)`。
参数 `perplexity`（困惑度，通常5-50）控制邻域规模，值越大考虑的邻居越多。

In [ ]:
# 模拟：4种音乐风格，每种 40 首，向量维度 64
np.random.seed(42)
n_per_class = 40
dim = 64
genres = ['古典', '爵士', '摇滚', '电子']
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

# 「训练后」embedding：每类有明确的中心，类内方差小
centers_trained = np.array([
    np.random.randn(dim) * 3 + np.array([10, 0] + [0] * (dim - 2)),
    np.random.randn(dim) * 3 + np.array([0, 10] + [0] * (dim - 2)),
    np.random.randn(dim) * 3 + np.array([-10, 0] + [0] * (dim - 2)),
    np.random.randn(dim) * 3 + np.array([0, -10] + [0] * (dim - 2)),
])
X_trained = np.vstack([
    centers_trained[i] + np.random.randn(n_per_class, dim) * 1.5
    for i in range(4)
])

# 「训练前」embedding：随机初始化，无结构
X_random = np.random.randn(n_per_class * 4, dim)

labels = np.repeat(np.arange(4), n_per_class)

# t-SNE 降维
tsne = TSNE(n_components=2, perplexity=20, random_state=0, n_iter=1000)
Z_trained = tsne.fit_transform(X_trained)
Z_random  = tsne.fit_transform(X_random)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, Z, title in zip(
    axes,
    [Z_random, Z_trained],
    ['随机初始化（无监督结构）', '对比学习后（类内聚集）']
):
    for i, (genre, c) in enumerate(zip(genres, colors)):
        mask = labels == i
        ax.scatter(Z[mask, 0], Z[mask, 1], c=c, label=genre,
                   alpha=0.75, s=40, edgecolors='none')
    ax.set_title(title, fontsize=12)
    ax.legend(loc='best', fontsize=9)
    ax.set_xlabel('t-SNE dim 1')
    ax.set_ylabel('t-SNE dim 2')
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('t-SNE：音乐 Embedding 聚类对比', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
print('✅ 左图：随机向量无结构散布；右图：对比学习后同类聚集成团')

## 3. 参数实验：`perplexity` 和 `n_iter` 对 t-SNE 的影响

| 参数 | 低值 | 高值 | 预期现象 |
|------|------|------|----------|
| `perplexity` | 5 | 50 | 低值→簇碎裂成小球；高值→簇拉伸融合，全局结构更清晰 |
| `n_iter` | 250 | 2000 | 迭代不足→形状未收敛，簇边界模糊；足够多→稳定分离 |
| `dim`（输入维度）| 8 | 256 | 高维且无对比学习→随机嵌入更难分离 |

实验建议：调整下方 `perplexity` 和 `n_iter`，观察四类音乐是否保持分离。

In [ ]:
# 参数实验：改变 perplexity，对比聚类效果
perplexity_list = [5, 20, 50]  # ← 修改这里
n_iter_exp = 500               # ← 也可以改这里（250 / 500 / 2000）

fig, axes = plt.subplots(1, len(perplexity_list),
                         figsize=(5 * len(perplexity_list), 4))
for ax, perp in zip(axes, perplexity_list):
    tsne_exp = TSNE(n_components=2, perplexity=perp,
                    random_state=0, n_iter=n_iter_exp)
    Z_exp = tsne_exp.fit_transform(X_trained)
    for i, (genre, c) in enumerate(zip(genres, colors)):
        mask = labels == i
        ax.scatter(Z_exp[mask, 0], Z_exp[mask, 1], c=c, label=genre,
                   alpha=0.8, s=35, edgecolors='none')
    ax.set_title(f'perplexity={perp}\nn_iter={n_iter_exp}', fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('t-SNE perplexity 参数实验（对比学习后 embedding）', y=1.02)
plt.tight_layout()
plt.show()

print(f'✅ 实验完成：perplexity={perplexity_list}，n_iter={n_iter_exp}')
print('预期：perplexity=5 簇碎裂；perplexity=50 簇边界更平滑但可能拉伸')

## 本节收束

本节用 `chroma` 矩阵可视化了和弦进行的时频结构，12行热力图直接"读出"了 I-IV-V-I 进行；
用 t-SNE 把 64维 `embed` 向量压到2D，对比了随机初始化与对比学习后的聚类效果。
这两张图是 `aurora.music` 模块（`chroma.py` + `embed.py`）的核心调试界面，
下一节（`mu4_recommend.ipynb`）将在已有 embedding 空间上实现余弦最近邻检索，把「看」变成「用」。